# The Seven Layers of AI: Layer 1: Classical AI

This notebook contains every example program from **Chapter 2** of the study guide *The Seven Layers of AI*, in the order they appear. Each program comes with the explanation that introduces it in the guide, the output printed in the guide, what to notice in that output, and the Investigate tasks.

**How to use it:** run the cells from top to bottom. Each program is self-contained, so you can also run any single program on its own after the setup cell. Change the code freely: the Investigate tasks suggest where to start.

## Programs in this notebook

1. `l1_expert.py`: an expert system with certainty factors
2. `l1_semantic.py`: a semantic network with inheritance
3. `l1_family.pl`: logic programming in Prolog
4. `l1_search.py`: four search algorithms compared
5. `l1_minimax.py`: perfect noughts and crosses
6. `l1_csp.py`: colouring a map of the British Isles
7. `l1_planner.py`: a delivery robot planner
8. `l1_fuzzy.py`: a fuzzy fan controller
9. `l1_flu_fever.py`: a two-node Bayesian network (Flu -> Fever)

## Setup

The programs in this notebook use: `numpy pgmpy`. If any are missing, uncomment the line below and run it once.

In [1]:
# %pip install numpy pgmpy
%matplotlib inline

## 1. An expert system with certainty factors

**File:** `l1_expert.py`

The program below implements every component of the architecture: rules held as data, working memory with certainty factors, forward and backward chaining engines, and an explanation trace. Read the comments carefully before running it.

In [2]:
"""
Layer 1 example: a rule-based expert system with certainty factors.
File: l1_expert.py
 
Purpose:
  Shows how 1970s and 1980s expert systems such as MYCIN captured a
  human specialist's know-how as IF-THEN rules, reasoned with uncertain
  evidence, and explained their conclusions. Here the "expert" diagnoses
  a computer that will not display anything.
 
How it works:
  * KNOWLEDGE BASE   -> the RULES list (data, not code)
  * WORKING MEMORY   -> the WorkingMemory class (facts and their certainty)
  * INFERENCE ENGINE -> forward_chain() and backward_chain()
  * EXPLANATION      -> the trace recorded for every rule that fires
 
  Certainty factors (CFs) follow the scheme used by MYCIN: a number from
  -1.0 (definitely false) to +1.0 (definitely true), with 0 meaning unknown.
 
What to look for in the output:
  * The trace shows WHICH rule fired and WHY, line by line.
  * memory_fault is supported by two rules (R5 and R6); its CF grows
    because the two pieces of evidence are combined.
  * Backward chaining starts from the goal and works down to observed
    facts, printing an indented proof tree.
 
Requires: standard library only.
"""
 
from dataclasses import dataclass, field
 
 
# ---------------------------------------------------------------------------
# 1. KNOWLEDGE BASE
# ---------------------------------------------------------------------------
# A Rule reads: IF all conditions are believed THEN believe the conclusion,
# with a strength (cf) that says how reliable the expert thinks the rule is.
@dataclass
class Rule:
    name: str          # a label so the explanation can refer to the rule
    conditions: list   # ALL of these facts must be believed (logical AND)
    conclusion: str    # the fact this rule supports
    cf: float          # how strongly the expert trusts the rule (0..1)
 
 
# The rules are plain data. A knowledge engineer can add, remove or edit
# them without touching the inference engine. This separation is what
# distinguishes an expert system from a program full of if-statements.
# Note that rules can chain: R1 concludes power_fault, which R2 then uses.
RULES = [
    Rule("R1", ["no_power_light"], "power_fault", 0.9),
    Rule("R2", ["power_fault", "cable_ok"], "psu_failure", 0.8),
    Rule("R3", ["power_light", "no_display"], "display_fault", 0.7),
    Rule("R4", ["display_fault", "monitor_works_elsewhere"],
         "graphics_card_fault", 0.85),
    # Two different rules reach the same conclusion: evidence will combine.
    Rule("R5", ["beeps_on_start"], "memory_fault", 0.6),
    Rule("R6", ["display_fault", "beeps_on_start"], "memory_fault", 0.75),
]
 
 
# ---------------------------------------------------------------------------
# 2. WORKING MEMORY
# ---------------------------------------------------------------------------
# Working memory is the system's "scratch pad" for one consultation: the
# facts observed so far plus everything inferred from them. The knowledge
# base stays fixed; working memory changes as reasoning proceeds.
@dataclass
class WorkingMemory:
    cf: dict = field(default_factory=dict)      # fact -> certainty factor
    trace: list = field(default_factory=list)   # explanation log
 
    def get(self, fact):
        return self.cf.get(fact, 0.0)           # unknown facts have CF 0
 
    def add(self, fact, new_cf, because):
        """Store evidence for a fact, COMBINING it with existing evidence.
 
        MYCIN's combination formula for two positive CFs is:
            combined = old + new * (1 - old)
        Two independent pieces of evidence therefore increase belief,
        but can never push it above 1.0.
        """
        old = self.get(fact)
        # First evidence for a fact is stored as is; later evidence only
        # closes part of the remaining gap between old and 1.0.
        combined = old + new_cf * (1 - old) if old > 0 else new_cf
        self.cf[fact] = round(combined, 3)
        # Recording the reason is the explanation facility: the system
        # can later answer "how did you reach that conclusion?"
        self.trace.append(f"{because}: {fact} CF {old:.2f} -> {combined:.2f}")
 
 
# ---------------------------------------------------------------------------
# 3a. INFERENCE ENGINE: FORWARD CHAINING (data driven)
# ---------------------------------------------------------------------------
# Forward chaining asks "given what I know, what else follows?". It suits
# situations where the data arrives first and we want every conclusion.
def forward_chain(wm, threshold=0.2):
    """Repeatedly fire every rule whose conditions are all believed.
 
    Starts from the facts and works towards conclusions. Stops when a full
    pass over the rules adds nothing new (a 'fixed point').
    """
    fired = set()                       # each rule fires once (refractoriness)
    changed = True
    # Keep sweeping the rules: a rule that fires may enable another rule
    # that was skipped earlier in the same pass.
    while changed:
        changed = False
        for rule in RULES:
            if rule.name in fired:
                continue
            cond_cfs = [wm.get(c) for c in rule.conditions]
            # A conjunction is only as strong as its weakest part,
            # so MYCIN uses the MINIMUM CF of the conditions.
            premise = min(cond_cfs)
            # The threshold stops very weak evidence from triggering rules.
            if premise >= threshold:
                # Belief in the conclusion = premise strength x rule strength.
                wm.add(rule.conclusion, premise * rule.cf,
                       because=(f"{rule.name} fired "
                                f"(premise {premise:.2f} x rule {rule.cf})"))
                fired.add(rule.name)
                changed = True
 
 
# ---------------------------------------------------------------------------
# 3b. INFERENCE ENGINE: BACKWARD CHAINING (goal driven)
# ---------------------------------------------------------------------------
# Backward chaining asks "could this hypothesis be true?". It only looks at
# rules relevant to the goal, so it asks fewer questions than forward
# chaining. Prolog (see l1_family.pl) reasons in exactly this way.
def backward_chain(goal, known, depth=0):
    """Try to PROVE a goal by finding rules that conclude it.
 
    If a condition is not already known, recursively try to prove it.
    If no rule concludes a condition, a real system would ASK THE USER;
    here we simply look it up in 'known' (a dict of observed facts).
    Returns True if the goal can be established.
    """
    pad = "  " * depth                  # indentation shows depth of the proof
    # Base case 1: the goal was observed directly.
    if known.get(goal):
        print(f"{pad}{goal}: observed fact")
        return True
    # Otherwise find every rule whose conclusion matches the goal.
    candidates = [r for r in RULES if r.conclusion == goal]
    # Base case 2: nothing to try, so the goal cannot be proved.
    if not candidates:
        print(f"{pad}{goal}: no rule concludes this and it was not observed")
        return False
    for rule in candidates:
        print(f"{pad}Trying {rule.name} to prove {goal}")
        # Each condition becomes a new sub-goal, proved the same way.
        if all(backward_chain(c, known, depth + 1) for c in rule.conditions):
            print(f"{pad}{goal}: PROVED by {rule.name}")
            return True
    # Every candidate rule failed: try elsewhere or give up.
    return False
 
 
if __name__ == "__main__":
    # Observations entered by the user, each with its own certainty
    # (e.g. 0.9 = "I am fairly sure the monitor works on another PC").
    wm = WorkingMemory()
    wm.add("power_light", 1.0, "user")
    wm.add("no_display", 1.0, "user")
    wm.add("monitor_works_elsewhere", 0.9, "user")
    wm.add("beeps_on_start", 0.8, "user")
 
    # Derive everything that follows, then show the explanation trace.
    print("=== FORWARD CHAINING ===")
    forward_chain(wm)
    for line in wm.trace:
        print(" ", line)
    # Ranking by CF gives the most likely diagnoses first.
    print("\nConclusions ranked by certainty:")
    for fact, cf in sorted(wm.cf.items(), key=lambda kv: -kv[1]):
        print(f"  {fact:25s} {cf:.2f}")
 
    # Now test one specific hypothesis instead of deriving everything.
    print("\n=== BACKWARD CHAINING: is it a graphics card fault? ===")
    observed = {"power_light": True, "no_display": True,
                "monitor_works_elsewhere": True}
    backward_chain("graphics_card_fault", observed)

=== FORWARD CHAINING ===
  user: power_light CF 0.00 -> 1.00
  user: no_display CF 0.00 -> 1.00
  user: monitor_works_elsewhere CF 0.00 -> 0.90
  user: beeps_on_start CF 0.00 -> 0.80
  R3 fired (premise 1.00 x rule 0.7): display_fault CF 0.00 -> 0.70
  R4 fired (premise 0.70 x rule 0.85): graphics_card_fault CF 0.00 -> 0.59
  R5 fired (premise 0.80 x rule 0.6): memory_fault CF 0.00 -> 0.48
  R6 fired (premise 0.70 x rule 0.75): memory_fault CF 0.48 -> 0.75

Conclusions ranked by certainty:
  power_light               1.00
  no_display                1.00
  monitor_works_elsewhere   0.90
  beeps_on_start            0.80
  memory_fault              0.75
  display_fault             0.70
  graphics_card_fault       0.59

=== BACKWARD CHAINING: is it a graphics card fault? ===
Trying R4 to prove graphics_card_fault
  Trying R3 to prove display_fault
    power_light: observed fact
    no_display: observed fact
  display_fault: PROVED by R3
  monitor_works_elsewhere: observed fact
graphics_ca

**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
=== FORWARD CHAINING ===
  user: power_light CF 0.00 -> 1.00
  user: no_display CF 0.00 -> 1.00
  user: monitor_works_elsewhere CF 0.00 -> 0.90
  user: beeps_on_start CF 0.00 -> 0.80
  R3 fired (premise 1.00 x rule 0.7): display_fault CF 0.00 -> 0.70
  R4 fired (premise 0.70 x rule 0.85): graphics_card_fault CF 0.00 -> 0.59
  R5 fired (premise 0.80 x rule 0.6): memory_fault CF 0.00 -> 0.48
  R6 fired (premise 0.70 x rule 0.75): memory_fault CF 0.48 -> 0.75
 
Conclusions ranked by certainty:
  power_light               1.00
  no_display                1.00
  monitor_works_elsewhere   0.90
  beeps_on_start            0.80
  memory_fault              0.75
  display_fault             0.70
  graphics_card_fault       0.59
 
=== BACKWARD CHAINING: is it a graphics card fault? ===
Trying R4 to prove graphics_card_fault
  Trying R3 to prove display_fault
    power_light: observed fact
    no_display: observed fact
  display_fault: PROVED by R3
  monitor_works_elsewhere: observed fact
graphics_card_fault: PROVED by R4
```

**What the output shows**

Notice three things in the output. First, memory_fault ends with a higher certainty (0.75) than either rule R5 or R6 would give alone, because evidence has been combined. Second, the trace is a complete, human-readable justification for every conclusion. Third, the backward chainer explored only the rules relevant to its goal, ignoring R1, R2, R5 and R6 entirely.

**Investigate**

1. Add a rule R7 concluding "overheating" from "fan_silent" and "random_shutdowns". Run the program with those facts. You changed no engine code: explain why that matters for maintenance.
2. Change the threshold in forward_chain() to 0.6. Which conclusions disappear and why?
3. Modify backward_chain() so that when no rule concludes a fact it asks the user with input() instead of looking it up. You have now built the consultation style of MYCIN.
4. Research why Yu et al. found MYCIN performed well, yet it was never used routinely in hospitals (Yu et al., 1979; Buchanan and Shortliffe, 1984). List the non-technical reasons.

## 2. A semantic network with inheritance

**File:** `l1_semantic.py`

In [3]:
"""
Layer 1 example: a semantic network with inheritance and exceptions.
File: l1_semantic.py
 
Purpose:
  Shows how classical AI represented general knowledge as a network of
  concepts linked by relations. Inheritance lets a system store a fact
  once (birds fly) and apply it to every kind of bird, while still
  allowing exceptions (penguins do not).
 
How it works:
  * Knowledge is stored as (subject, relation, object) triples, the same
    structure used today by knowledge graphs and the RDF standard.
  * 'is_a' links create a hierarchy (tweety -> canary -> bird -> animal).
  * A property not stored on a concept is INHERITED from its ancestors,
    unless a more specific concept overrides it.
  * lookup() climbs the hierarchy and stops at the nearest answer.
 
What to look for in the output:
  * tweety can fly (inherited from bird) but pingu cannot (the penguin
    exception is found first, so it wins).
  * pingu has feathers, found two levels up the hierarchy.
  * pingu's colour is unknown: nothing on its path records one.
  * The "found via" path shows exactly how each answer was reached.
 
Requires: standard library only.
"""
 
# The whole knowledge base. Each triple is one link in the network.
TRIPLES = [
    ("animal", "has", "skin"),
    ("bird", "is_a", "animal"),
    ("bird", "can_fly", True),           # a DEFAULT for all birds
    ("bird", "has", "feathers"),
    ("canary", "is_a", "bird"),
    ("canary", "colour", "yellow"),
    ("penguin", "is_a", "bird"),
    ("penguin", "can_fly", False),       # exception overrides the default
    # Individuals sit at the bottom of the hierarchy.
    ("tweety", "is_a", "canary"),
    ("pingu", "is_a", "penguin"),
]
 
 
# Facts stored directly on a concept, with no inheritance.
def direct(subject, relation):
    return [o for s, r, o in TRIPLES if s == subject and r == relation]
 
 
# Inheritance: if a concept does not know the answer itself, ask its
# parent, then its grandparent, and so on. Because the most specific
# concept is checked first, exceptions automatically beat defaults.
def lookup(subject, relation, path=None):
    """Search upward through is_a links; the nearest answer wins."""
    path = (path or []) + [subject]       # remember the route for explanation
    values = direct(subject, relation)
    if values:                            # answer stored here: stop climbing
        return values, " -> ".join(path)
    # No local answer, so recurse into each parent concept.
    for parent in direct(subject, "is_a"):
        found = lookup(parent, relation, path)
        if found[0]:
            return found
    # Reached the top of the hierarchy without finding anything.
    return [], " -> ".join(path)
 
 
# Every concept above this one: its parents, their parents, and so on.
def ancestors(subject):
    parents = direct(subject, "is_a")
    return parents + [a for p in parents for a in ancestors(p)]
 
 
if __name__ == "__main__":
    # Each query asks: what is the value of this relation for this entity?
    for who, rel in [("tweety", "can_fly"), ("pingu", "can_fly"),
                     ("pingu", "has"), ("tweety", "colour"), ("pingu", "colour")]:
        value, path = lookup(who, rel)
        print(f"{who:7s} {rel:8s} = {str(value or 'unknown'):<14} found via {path}")
    print("Ancestors of tweety:", ancestors("tweety"))

tweety  can_fly  = [True]         found via tweety -> canary -> bird
pingu   can_fly  = [False]        found via pingu -> penguin
pingu   has      = ['feathers']   found via pingu -> penguin -> bird
tweety  colour   = ['yellow']     found via tweety -> canary
pingu   colour   = unknown        found via pingu
Ancestors of tweety: ['canary', 'bird', 'animal']


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
tweety  can_fly  = [True]         found via tweety -> canary -> bird
pingu   can_fly  = [False]        found via pingu -> penguin
pingu   has      = ['feathers']   found via pingu -> penguin -> bird
tweety  colour   = ['yellow']     found via tweety -> canary
pingu   colour   = unknown        found via pingu
Ancestors of tweety: ['canary', 'bird', 'animal']
```

**Investigate**

1. Add an "ostrich" that is a bird and cannot fly, and a "bat" that is an animal and can fly. Test both.
2. Add a second parent so that "pingu" IS-A "penguin" and IS-A "cartoon_character", where cartoon characters can_fly is True. What does the program answer? This is the "Nixon diamond" problem of multiple inheritance with conflicting defaults.
3. Compare this code with a Python class hierarchy (class Bird(Animal)). Which features correspond?

## 3. Logic programming in Prolog

**File:** `l1_family.pl`

Install SWI-Prolog from swi-prolog.org (or use the online SWISH environment at swish.swi-prolog.org), save the file below, then load it and type the queries shown in the comments.

> **Not run in this notebook.** This program needs a Prolog interpreter such as SWI-Prolog. Install what it needs and run it on your own machine; the code is included so that you can read and adapt it.

In [4]:
%%writefile l1_family.pl
% Layer 1 example: logic programming in Prolog.
% File: l1_family.pl
%
% Purpose:
%   In logic programming you describe WHAT is true (facts and rules)
%   and the system works out HOW to answer questions by logical
%   inference. Prolog was a leading AI language of the 1970s and 80s.
%
% How it works:
%   * Facts record parent relationships.
%   * Rules define grandparent, ancestor and sibling from those facts.
%   * A query is answered by backward chaining: Prolog tries to prove
%     it, matching rules and facts, and backtracks to find more answers.
%
% What to look for in the output:
%   * One query can return several answers, one per ';' you type.
%   * ancestor/2 is recursive, so it works for any number of generations.
%   * Rules were never told who george's ancestors are: they are derived.
%
% Requires: SWI-Prolog.
% Run with SWI-Prolog:  swipl l1_family.pl   then type queries at ?-
%
% FACTS state what is true. Arguments starting with lower case are constants.
% parent(X, Y) reads "X is a parent of Y".
parent(elizabeth, charles).
parent(philip,    charles).
parent(charles,   william).
parent(charles,   harry).
parent(william,   george).
 
% RULES define new relations. Variables start with an upper case letter.
% ':-' is read as 'if' and ',' as 'and'.
% G is a grandparent of C if G is a parent of some P who is a parent of C.
grandparent(G, C) :- parent(G, P), parent(P, C).
 
% Recursive rule: X is an ancestor of Y if X is a parent of Y,
% or X is a parent of someone who is an ancestor of Y.
% The first clause is the base case that stops the recursion.
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).
 
% Two people are siblings if they share a parent. A \= B stops a person
% being counted as their own sibling.
sibling(A, B) :- parent(P, A), parent(P, B), A \= B.
 
% Example queries (Prolog answers by BACKWARD CHAINING and backtracking):
% ?- grandparent(elizabeth, Who).     finds william, then harry on ';'
% ?- ancestor(Who, george).           lists every ancestor of george
% ?- sibling(william, X).             X = harry

Writing l1_family.pl


The cell above saves the Prolog program to a file. To run it, install SWI-Prolog (https://www.swi-prolog.org), then in a terminal run `swipl l1_family.pl` and type queries such as `grandparent(X, Y).` at the `?-` prompt.

**Investigate**

1. Run the query ancestor(Who, george). and press ; repeatedly to see every answer. Draw the search tree Prolog explored.
2. Swap the order of the two ancestor rules so the recursive rule is first, then move the recursive call to the start of its rule: ancestor(X, Y) :- ancestor(Z, Y), parent(X, Z). What happens and why? This demonstrates that Prolog's search order is not purely logical.
3. Write rules for cousin(A, B) and uncle_or_aunt(A, B).

## 4. Four search algorithms compared

**File:** `l1_search.py`

The program solves the same grid problem with BFS, DFS, uniform-cost search and A\*. Mud squares cost 5 to enter, so the shortest route in steps is not the cheapest route. The whole difference between the four algorithms is a single line that chooses which state to expand next.

In [5]:
"""
Layer 1 example: comparing uninformed and informed search on a grid.
File: l1_search.py
 
Purpose:
  Many AI problems (route finding, puzzles, planning) can be framed as
  searching a space of states for a path to a goal. This program shows
  that four famous algorithms are really ONE algorithm, and that a good
  heuristic lets A* find the cheapest route while exploring less.
 
How it works:
  Four algorithms share ONE generic function. The only difference between
  them is how the FRONTIER (the list of states waiting to be explored)
  decides which state to expand next:
 
    BFS     -> first in, first out queue       (shallowest state first)
    DFS     -> last in, first out stack        (deepest state first)
    UCS     -> priority queue ordered by g     (cheapest path so far first)
    A*      -> priority queue ordered by g + h (cheapest estimated total)
 
  g(n) = actual cost from the start to state n
  h(n) = heuristic ESTIMATE of cost from n to the goal
 
What to look for in the output:
  * BFS and DFS find paths with few steps but a high cost, because they
    ignore the expensive mud.
  * UCS and A* both find the cheapest route (cost 18).
  * A* expands the fewest states: the heuristic steers it to the goal.
  * The drawn route goes the long way round rather than through mud.
 
Requires: standard library only.
"""
 
import heapq
from collections import deque
 
# The STATE SPACE: each open cell is a state; S is the start, G the goal.
# '#' is a wall, '~' is mud (costs 5 to enter), '.' costs 1
GRID = [
    "S....#........",
    ".##..#.~~~~~..",
    "..#..#.~###~..",
    "..#....~#G#~..",
    "..####.~#.#~..",
    "......#~~.~~..",
    "..............",
]
 
ROWS, COLS = len(GRID), len(GRID[0])
# Locate a character in the grid and return its (row, column) state.
find = lambda ch: next((r, c) for r in range(ROWS) for c in range(COLS)
                       if GRID[r][c] == ch)
START, GOAL = find("S"), find("G")
 
 
# The successor function defines the problem: from a state, which states
# can we reach in one move, and what does each move cost?
def neighbours(state):
    """Successor function: legal moves up, down, left and right."""
    r, c = state
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        # Stay inside the grid and never step into a wall.
        if 0 <= nr < ROWS and 0 <= nc < COLS and GRID[nr][nc] != "#":
            step_cost = 5 if GRID[nr][nc] == "~" else 1
            yield (nr, nc), step_cost
 
 
# A heuristic is a cheap guess at the remaining cost. It must be
# ADMISSIBLE (never an overestimate): if A* were misled into thinking a
# good route was expensive it might skip it and return a worse one.
def manhattan(state):
    """Admissible heuristic: never overestimates, because every move
    costs at least 1 and you cannot reach the goal in fewer moves than
    the horizontal plus vertical distance."""
    return abs(state[0] - GOAL[0]) + abs(state[1] - GOAL[1])
 
 
# One search loop for all four strategies: take a state from the frontier,
# stop if it is the goal, otherwise add its neighbours to the frontier.
def search(strategy):
    """Generic graph search. Returns (path, cost, number of expansions)."""
    counter = 0                                   # tie breaker for the heap
    # BFS and DFS use a plain double-ended queue; UCS and A* use a heap
    # so the lowest-priority state can always be removed first.
    if strategy in ("BFS", "DFS"):
        frontier = deque([START])
    else:
        frontier = [(0, counter, START)]          # (priority, tie, state)
    came_from = {START: None}                     # remembers how we got here
    g = {START: 0}                                # best known cost to state
    expanded = 0                                  # measures the work done
 
    while frontier:
        # ---- choose next state: THIS LINE IS THE WHOLE DIFFERENCE ----
        if strategy == "BFS":
            state = frontier.popleft()            # oldest state (queue)
        elif strategy == "DFS":
            state = frontier.pop()                # newest state (stack)
        else:
            _, _, state = heapq.heappop(frontier) # lowest priority value
        expanded += 1
 
        # Testing when a state is EXPANDED (not when it is first seen)
        # is what guarantees UCS and A* return the cheapest path.
        if state == GOAL:                         # goal test on expansion
            # Rebuild the path by following came_from links backwards.
            path = []
            while state:
                path.append(state)
                state = came_from[state]
            return path[::-1], g[GOAL], expanded
 
        for nxt, cost in neighbours(state):
            new_g = g[state] + cost
            if strategy in ("BFS", "DFS"):
                # Uninformed: each state is added once, when first seen.
                if nxt not in came_from:          # ignore costs entirely
                    came_from[nxt], g[nxt] = state, new_g
                    frontier.append(nxt)
            elif nxt not in g or new_g < g[nxt]:  # found a cheaper route
                g[nxt], came_from[nxt] = new_g, state
                # UCS orders by cost so far (h = 0); A* adds the estimate.
                priority = new_g + (manhattan(nxt) if strategy == "A*" else 0)
                counter += 1
                heapq.heappush(frontier, (priority, counter, nxt))
    # Frontier exhausted: the goal is unreachable.
    return None, None, expanded
 
 
# Show the route on the map, marking each intermediate cell with '*'.
def draw(path):
    rows = [list(row) for row in GRID]
    for r, c in path[1:-1]:                       # keep S and G visible
        rows[r][c] = "*"
    return "\n".join("".join(row) for row in rows)
 
 
if __name__ == "__main__":
    # Run all four strategies on the same problem and compare them.
    print(f"{'Algorithm':8s} {'Steps':>6s} {'Cost':>6s} {'Expanded':>9s}")
    for s in ["BFS", "DFS", "UCS", "A*"]:
        path, cost, n = search(s)
        print(f"{s:8s} {len(path)-1:6d} {cost:6d} {n:9d}")
    print("\nA* route (* marks the path):")
    print(draw(search("A*")[0]))

Algorithm  Steps   Cost  Expanded
BFS          16     28        59
DFS          16     32        53
UCS          18     18        55
A*           18     18        49

A* route (* marks the path):
S....#........
*##..#.~~~~~..
*.#..#.~###~..
**#....~#G#~..
.*####.~#*#~..
.*****#~~*~~..
.....*****....


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Algorithm  Steps   Cost  Expanded
BFS          16     28        59
DFS          16     32        53
UCS          18     18        55
A*           18     18        49
 
A* route (* marks the path):
S....#........
*##..#.~~~~~..
*.#..#.~###~..
**#....~#G#~..
.*####.~#*#~..
.*****#~~*~~..
.....*****....
```

**What the output shows**

The results confirm the theory. BFS finds the route with the fewest steps but walks through mud, costing 28. DFS finds an even more expensive route. Uniform-cost search and A\* both find the cheapest route (cost 18), because they take step costs into account, and A\* expands the fewest states because the Manhattan heuristic guides it towards the goal.

**Investigate**

1. Multiply the heuristic by 3 inside the A\* priority calculation. The heuristic is no longer admissible. Does A\* still find the cheapest route? How many states does it expand? This trade-off is called weighted A\*.
2. Enlarge the grid to 50 by 50 with random walls. Plot the number of expansions for each algorithm.
3. Implement iterative deepening search and add it to the comparison.

## 5. Perfect noughts and crosses

**File:** `l1_minimax.py`

In [6]:
"""
Layer 1 example: adversarial search for noughts and crosses (tic-tac-toe).
File: l1_minimax.py
 
Purpose:
  In games the world contains an opponent who is working against you.
  Minimax chooses moves by assuming the opponent will always reply with
  their best move. Alpha-beta pruning gets the same answer far faster,
  the idea behind classic game programs such as Deep Blue.
 
How it works:
  Minimax assumes both players play perfectly:
    * MAX ('X') picks the move with the HIGHEST score
    * MIN ('O') picks the move with the LOWEST score
  Scores: +1 X wins, -1 O wins, 0 draw.
 
  Alpha-beta pruning returns exactly the same answer as plain minimax, but
  skips branches that cannot change the decision:
    alpha = best score MAX can already guarantee
    beta  = best score MIN can already guarantee
  If alpha >= beta, the opponent would never allow this line, so stop.
 
What to look for in the output:
  * Both methods choose the same opening move.
  * Alpha-beta examines about 30 thousand positions instead of about
    550 thousand: the same answer for under a tenth of the work.
  * Every opening scores 0: with perfect play the game is a draw.
  * In the mid-game position, X spots the immediate win at square 2.
 
Requires: standard library only.
"""
 
import math
 
# Squares are numbered 0-8, left to right, top to bottom.
# These are the eight rows, columns and diagonals that win the game.
WIN_LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
nodes = 0   # counts positions examined, so we can measure pruning
 
 
# Terminal test: returns 'X' or 'O' for a win, 'draw' for a full board,
# or None if the game is still in progress.
def winner(board):
    for a, b, c in WIN_LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    return "draw" if " " not in board else None
 
 
# Minimax looks ahead to the end of the game. The value of a position is
# the best score the player to move can force, assuming the other player
# also picks their best reply at every level below.
def minimax(board, is_max):
    """Plain minimax: explores the ENTIRE game tree below this position."""
    global nodes
    nodes += 1
    result = winner(board)
    if result:                                   # terminal state reached
        # The utility function: how good this final position is for X.
        return {"X": 1, "O": -1, "draw": 0}[result]
    scores = []
    for i in range(9):
        if board[i] == " ":
            board[i] = "X" if is_max else "O"    # make the move
            # The opponent moves next, so switch between MAX and MIN.
            scores.append(minimax(board, not is_max))
            board[i] = " "                       # undo the move (backtrack)
    # MAX takes the highest child score, MIN the lowest.
    return max(scores) if is_max else min(scores)
 
 
# Alpha-beta carries two bounds down the tree. Once MIN has a reply that
# makes this position worse for MAX than something MAX already has
# elsewhere, the remaining replies cannot matter, so they are skipped.
def alphabeta(board, is_max, alpha=-math.inf, beta=math.inf):
    """Minimax with alpha-beta pruning."""
    global nodes
    nodes += 1
    result = winner(board)
    if result:
        return {"X": 1, "O": -1, "draw": 0}[result]
    # Start from the worst possible score for the player to move.
    best = -math.inf if is_max else math.inf
    for i in range(9):
        if board[i] != " ":
            continue
        board[i] = "X" if is_max else "O"
        score = alphabeta(board, not is_max, alpha, beta)
        board[i] = " "
        if is_max:
            best = max(best, score)
            alpha = max(alpha, best)             # MAX's guarantee improves
        else:
            best = min(best, score)
            beta = min(beta, best)               # MIN's guarantee improves
        # The window has closed: the player above would never choose
        # this position, so exploring more moves here is wasted effort.
        if alpha >= beta:                        # PRUNE: skip remaining moves
            break
    return best
 
 
# The top-level decision: score each legal move by searching the game
# tree beneath it (O replies next, so is_max is False).
def best_move(board, use_pruning=True):
    """Try every legal move for X and pick the one with the best score."""
    fn = alphabeta if use_pruning else minimax
    moves = {i: None for i in range(9) if board[i] == " "}
    for i in moves:
        board[i] = "X"
        moves[i] = fn(board, False)
        board[i] = " "
    # Ties go to the lowest-numbered square.
    return max(moves, key=moves.get), moves
 
 
if __name__ == "__main__":
    # Solve the empty board twice and count the positions each method needs.
    empty = [" "] * 9
    for label, prune in [("Plain minimax", False), ("Alpha-beta", True)]:
        nodes = 0
        move, scores = best_move(empty[:], prune)
        print(f"{label:14s}: best opening square {move}, positions examined {nodes:,}")
    print("Scores for each opening square (0 = perfect play leads to a draw):", scores)
 
    # A mid-game position: X to move and can win immediately at square 2
    board = list("XX OO    ")
    move, _ = best_move(board)
    print("\nPosition:", board, "-> X should play square", move)

Plain minimax : best opening square 0, positions examined 549,945
Alpha-beta    : best opening square 0, positions examined 30,709
Scores for each opening square (0 = perfect play leads to a draw): {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}

Position: ['X', 'X', ' ', 'O', 'O', ' ', ' ', ' ', ' '] -> X should play square 2


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Plain minimax : best opening square 0, positions examined 549,945
Alpha-beta    : best opening square 0, positions examined 30,709
Scores for each opening square (0 = perfect play leads to a draw): {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}
 
Position: ['X', 'X', ' ', 'O', 'O', ' ', ' ', ' ', ' '] -> X should play square 2
```

**What the output shows**

The output shows the evidence clearly: both methods choose a move scoring 0 (with perfect play noughts and crosses is always a draw), but alpha-beta examined around 30,700 positions compared with around 550,000, an 18-fold saving with identical results.

**Investigate**

1. Change the move loop in alphabeta() to try the centre (4) first, then corners, then edges. How many positions are examined now? Explain using Knuth and Moore's result on move ordering.
2. Add a simple text interface so you can play against the program. Try to beat it.
3. Add a depth limit and an evaluation function that counts lines still open to each player. What happens to play quality at depth 2?

## 6. Colouring a map of the British Isles

**File:** `l1_csp.py`

In [7]:
"""
Layer 1 example: constraint satisfaction.
File: l1_csp.py
 
Purpose:
  Many real tasks (timetabling, scheduling, puzzles such as Sudoku) are
  about finding values that satisfy a set of rules. A constraint solver
  searches systematically and can PROVE when no solution exists, which
  statistical or learned methods cannot.
 
How it works:
  A CSP has:
    VARIABLES   things we must assign (regions of a map)
    DOMAINS     allowed values for each variable (colours)
    CONSTRAINTS rules the assignment must obey (neighbours differ)
 
  We solve map colouring for a simplified map of UK and Irish nations and
  regions using BACKTRACKING SEARCH improved by two classic techniques:
    * MRV heuristic (minimum remaining values): assign the most
      constrained variable first, so failures are discovered early
    * FORWARD CHECKING: after each assignment, remove that value from
      neighbours' domains; if any domain becomes empty, backtrack at once
 
What to look for in the output:
  * With 3 colours the solver reports NO SOLUTION after trying every
    option: a proof of impossibility.
  * With 4 colours it succeeds in only 6 value trials, with no dead ends.
  * The final check confirms that no two neighbours share a colour.
 
Requires: standard library only.
"""
 
# The constraint graph: each region lists the regions it must differ from.
NEIGHBOURS = {
    "Scotland":         ["England", "NorthernIreland"],   # sea link counted
    "England":          ["Scotland", "Wales"],
    "Wales":            ["England", "Ireland"],           # sea link counted
    "NorthernIreland":  ["Ireland", "Scotland"],
    "Ireland":          ["NorthernIreland", "Wales"],
    "IsleOfMan":        ["England", "Scotland", "NorthernIreland", "Wales", "Ireland"],
}
# make the relationship symmetric (if A borders B then B borders A)
for a, ns in list(NEIGHBOURS.items()):
    for b in ns:
        NEIGHBOURS.setdefault(b, [])
        if a not in NEIGHBOURS[b]:
            NEIGHBOURS[b].append(a)
 
ALL_COLOURS = ["red", "green", "blue", "yellow"]
steps = 0          # counts value trials, a measure of search effort
 
 
# Constraint check: is this colour allowed given the colours already placed?
def consistent(var, value, assignment):
    """The only constraint: adjacent regions must have different colours."""
    # Unassigned neighbours return None, which never clashes.
    return all(assignment.get(n) != value for n in NEIGHBOURS[var])
 
 
# Backtracking search: assign one variable at a time; if a choice leads to
# a dead end, undo it and try the next value. 'domains' holds the colours
# still possible for each region after forward checking.
def backtrack(assignment, domains):
    global steps
    if len(assignment) == len(NEIGHBOURS):          # every variable assigned
        return assignment
 
    # MRV: choose the unassigned variable with the fewest legal values left
    # (the "fail first" principle: tackle the hardest choice while cheap).
    var = min((v for v in NEIGHBOURS if v not in assignment),
              key=lambda v: len(domains[v]))
 
    for value in domains[var]:
        steps += 1
        if not consistent(var, value, assignment):
            continue
        assignment[var] = value
 
        # FORWARD CHECKING: prune this colour from unassigned neighbours
        pruned = {n: [c for c in domains[n] if c != value]
                  for n in NEIGHBOURS[var] if n not in assignment}
        # An empty domain means some neighbour now has no colour left, so
        # this branch is doomed; skip it without searching any deeper.
        if all(pruned.values()):                     # no neighbour left empty
            # Copy the domains so undoing a choice restores the old ones.
            new_domains = {**domains, **pruned}
            result = backtrack(assignment, new_domains)
            if result:
                return result
 
        del assignment[var]                          # undo and try next value
    return None                                      # triggers backtracking
 
 
if __name__ == "__main__":
    # Try with 3 colours first, then 4. The five mainland regions form a
    # ring of odd length AND the Isle of Man touches all of them, so three
    # colours are mathematically impossible. Backtracking PROVES this by
    # exhausting every option, a guarantee a learned model cannot give.
    for k in (3, 4):
        steps = 0
        colours = ALL_COLOURS[:k]
        # Every region starts with the full set of k colours available.
        solution = backtrack({}, {v: colours[:] for v in NEIGHBOURS})
        print(f"{k} colours: {'solved' if solution else 'NO SOLUTION'} "
              f"after {steps} value trials")
    print("Four-colour solution:")
    for region, colour in solution.items():
        print(f"  {region:16s} {colour}")
    # Verify the solution independently
    ok = all(solution[a] != solution[b] for a in NEIGHBOURS for b in NEIGHBOURS[a])
    print("All constraints satisfied:", ok)

3 colours: NO SOLUTION after 27 value trials
4 colours: solved after 6 value trials
Four-colour solution:
  Scotland         red
  England          green
  IsleOfMan        blue
  Wales            red
  NorthernIreland  green
  Ireland          yellow
All constraints satisfied: True


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
3 colours: NO SOLUTION after 27 value trials
4 colours: solved after 6 value trials
Four-colour solution:
  Scotland         red
  England          green
  IsleOfMan        blue
  Wales            red
  NorthernIreland  green
  Ireland          yellow
All constraints satisfied: True
```

**What the output shows**

This example makes an important point about classical AI. With three colours the solver does not just fail to find an answer; it exhaustively proves that no answer exists. A learned model can only say that it did not find one. Guarantees like this are why constraint solvers and logic are still used where correctness must be demonstrated.

**Investigate**

1. Remove the MRV heuristic (choose variables in fixed alphabetical order) and compare the number of value trials.
2. Adapt the program to solve a 4 by 4 Sudoku: 16 variables, domain 1 to 4, and constraints for rows, columns and 2 by 2 boxes.
3. Install OR-Tools (pip install ortools) and solve the same map using its CP-SAT solver. Compare the amount of code needed.

## 7. A delivery robot planner

**File:** `l1_planner.py`

In [8]:
"""
Layer 1 example: a STRIPS-style automated planner.
File: l1_planner.py
 
Purpose:
  Planning means working out a sequence of actions that turns the current
  situation into one where a goal holds. STRIPS (Fikes and Nilsson, 1971)
  was built for the Shakey robot, and its way of describing actions is
  still the basis of planning languages such as PDDL.
 
How it works:
  In STRIPS the world is a SET of true facts.
  Each action has:
    pre  facts that must be true before the action can be taken
    add  facts the action makes true
    dele facts the action makes false
  Everything not mentioned stays the same. This simplifying assumption is
  one practical answer to the 'frame problem'.
 
  Planning is then SEARCH through world states, reusing the idea of BFS.
  Domain: a robot must move a parcel from the depot to the office.
 
What to look for in the output:
  * The robot first travels to the depot, since it cannot pick up the
    parcel from a distance (a precondition).
  * The plan ends by returning to the depot, because the goal also
    requires robot_at_depot.
  * Only 12 world states are explored: BFS guarantees the shortest plan.
 
Requires: standard library only.
"""
 
from collections import deque
from dataclasses import dataclass
 
 
# An action schema. frozenset makes the fact sets fast to compare and
# lets whole world states be stored in a 'visited' set.
@dataclass(frozen=True)
class Action:
    name: str
    pre: frozenset
    add: frozenset
    dele: frozenset       # 'del' is a Python keyword, hence 'dele'
 
 
# Shorthand constructor that turns ordinary sets into frozensets.
def A(name, pre, add, dele):
    return Action(name, frozenset(pre), frozenset(add), frozenset(dele))
 
 
# The domain: three places joined in a line, depot - corridor - office.
PLACES = ["depot", "corridor", "office"]
LINKS = [("depot", "corridor"), ("corridor", "office")]
 
# Generate every concrete action. For example move(depot,corridor) needs
# robot_at_depot, adds robot_at_corridor and deletes robot_at_depot.
ACTIONS = []
for a, b in LINKS + [(y, x) for x, y in LINKS]:     # movement in both directions
    ACTIONS.append(A(f"move({a},{b})", {f"robot_at_{a}"},
                     {f"robot_at_{b}"}, {f"robot_at_{a}"}))
for p in PLACES:
    # Pick up: robot and parcel in the same place, and the gripper free.
    ACTIONS.append(A(f"pickup(parcel,{p})",
                     {f"robot_at_{p}", f"parcel_at_{p}", "hand_empty"},
                     {"holding_parcel"}, {f"parcel_at_{p}", "hand_empty"}))
    # Drop: the parcel appears wherever the robot currently is.
    ACTIONS.append(A(f"drop(parcel,{p})",
                     {f"robot_at_{p}", "holding_parcel"},
                     {f"parcel_at_{p}", "hand_empty"}, {"holding_parcel"}))
 
 
# Forward state-space planning: start from the initial state, try every
# applicable action, and search breadth first until a state satisfies
# the goal. Each search node stores the state and the plan that reached it.
def plan(initial, goal):
    """Breadth-first search over sets of facts; returns shortest plan."""
    start = frozenset(initial)
    frontier = deque([(start, [])])
    visited = {start}                              # avoid revisiting states
    while frontier:
        state, steps = frontier.popleft()
        # The goal is only partly specified: any state containing all the
        # goal facts will do, whatever else is true.
        if goal <= state:                          # all goal facts are true
            return steps, len(visited)
        for act in ACTIONS:
            if act.pre <= state:                   # action is applicable
                # Result = old facts, minus the delete list, plus the add list.
                nxt = (state - act.dele) | act.add # apply STRIPS update rule
                if nxt not in visited:
                    visited.add(nxt)
                    frontier.append((nxt, steps + [act.name]))
    # No sequence of actions reaches the goal.
    return None, len(visited)
 
 
if __name__ == "__main__":
    # The robot starts in the office; the parcel waits at the depot.
    initial = {"robot_at_office", "parcel_at_depot", "hand_empty"}
    goal = {"parcel_at_office", "robot_at_depot"}
    steps, explored = plan(initial, goal)
    print(f"Plan found ({explored} states explored):")
    for i, s in enumerate(steps, 1):
        print(f"  {i}. {s}")

Plan found (12 states explored):


  1. move(office,corridor)
  2. move(corridor,depot)
  3. pickup(parcel,depot)
  4. move(depot,corridor)
  5. move(corridor,office)
  6. drop(parcel,office)
  7. move(office,corridor)
  8. move(corridor,depot)


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Plan found (12 states explored):
  1. move(office,corridor)
  2. move(corridor,depot)
  3. pickup(parcel,depot)
  4. move(depot,corridor)
  5. move(corridor,office)
  6. drop(parcel,office)
  7. move(office,corridor)
  8. move(corridor,depot)
```

**Investigate**

1. Add a "lobby" location linked to the corridor and a second parcel. Set a goal requiring both parcels in the office. How does the number of explored states grow?
2. Remove "hand_empty" from the pickup preconditions. What unrealistic plans can the robot now produce? This shows why careful domain modelling matters.
3. Replace BFS with A\* using the number of unsatisfied goal facts as the heuristic. Is this heuristic admissible?

## 8. A fuzzy fan controller

**File:** `l1_fuzzy.py`

In [9]:
"""
Layer 1 example: a fuzzy logic fan controller (Zadeh, 1965; Mamdani, 1975).
File: l1_fuzzy.py
 
Purpose:
  Classical logic says a statement is TRUE (1) or FALSE (0).
  Fuzzy logic lets it be PARTLY true, for example 24 C is 0.6 'warm'.
  This lets vague human rules ("if it is warm and humid, run the fan
  fast") drive smooth control, as in washing machines and air conditioning.
 
How it works:
  The controller works in four stages:
    1. FUZZIFY    convert crisp inputs into degrees of membership
    2. RULES      apply IF-THEN rules using fuzzy AND (min) and OR (max)
    3. AGGREGATE  combine the clipped output sets (max)
    4. DEFUZZIFY  turn the combined shape back into one number (centroid)
 
What to look for in the output:
  * Several rules can be partly true at once (24 C, 75% humidity fires
    both medium and fast), and the result blends them.
  * Fan speed changes smoothly with temperature, with no sudden jumps.
  * Humidity matters: at 24 C, damp air pushes the fan above 50%.
 
Requires: numpy
"""
 
import numpy as np
 
 
# A membership function maps a crisp value to a degree between 0 and 1.
# A triangle is the simplest shape: 0 at a, 1 at the peak b, 0 again at c.
# The tiny 1e-9 avoids division by zero when two corners coincide.
def tri(x, a, b, c):
    """Triangular membership function rising from a, peaking at b, falling to c."""
    return np.maximum(np.minimum((x - a) / (b - a + 1e-9), (c - x) / (c - b + 1e-9)), 0)
 
 
# Universe of discourse for the output: fan speed 0 to 100 percent
# (sampled at 501 points so the shapes can be combined numerically).
speed = np.linspace(0, 100, 501)
# Output fuzzy sets. They overlap, so a speed can be partly slow and
# partly medium, just as people's categories overlap.
SPEED_SETS = {
    "slow":   tri(speed, -1, 0, 50),
    "medium": tri(speed, 25, 50, 75),
    "fast":   tri(speed, 50, 100, 101),
}
 
 
# One full pass of a Mamdani controller: crisp inputs in, crisp output out.
def controller(temp_c, humidity):
    # ---- 1. FUZZIFY -------------------------------------------------------
    # Each input temperature belongs to each set to some degree. Because
    # the triangles overlap, 24 C is partly warm AND slightly hot.
    cold = float(tri(temp_c, 0, 10, 20))
    warm = float(tri(temp_c, 15, 22, 30))
    hot = float(tri(temp_c, 25, 35, 45))
    # Humidity uses a ramp instead of a triangle.
    humid = float(np.clip((humidity - 40) / 40, 0, 1))  # 40% -> 0, 80% -> 1
 
    # ---- 2. RULE EVALUATION ----------------------------------------------
    # Each rule's strength is how true its condition is.
    # Fuzzy AND takes the minimum, fuzzy OR takes the maximum.
    rules = {
        "slow":   cold,                          # IF cold THEN slow
        "medium": warm,                          # IF warm THEN medium
        # IF hot OR (warm AND humid) THEN fast
        "fast":   max(hot, min(warm, humid)),
    }
 
    # ---- 3. AGGREGATION ---------------------------------------------------
    # Clip each output set at its rule strength, then take the maximum.
    # A rule that is 0.75 true can only "fill" its set up to height 0.75.
    combined = np.zeros_like(speed)
    for label, strength in rules.items():
        combined = np.maximum(combined, np.minimum(SPEED_SETS[label], strength))
 
    # ---- 4. DEFUZZIFICATION (centre of gravity) --------------------------
    # The balance point of the combined shape: a weighted average of all
    # speeds, weighted by how strongly each speed is supported.
    if combined.sum() == 0:                      # no rule fired at all
        return 0.0, rules
    return float((speed * combined).sum() / combined.sum()), rules
 
 
if __name__ == "__main__":
    # Four test conditions from cool and dry to hot.
    for t, h in [(8, 30), (21, 30), (24, 75), (33, 50)]:
        out, rules = controller(t, h)
        strengths = ", ".join(f"{k}={v:.2f}" for k, v in rules.items())
        print(f"{t:>3} C, {h:>3}% humidity -> fan {out:5.1f}%   ({strengths})")

  8 C,  30% humidity -> fan  17.2%   (slow=0.80, medium=0.00, fast=0.00)
 21 C,  30% humidity -> fan  50.0%   (slow=0.00, medium=0.86, fast=0.00)
 24 C,  75% humidity -> fan  66.5%   (slow=0.00, medium=0.75, fast=0.75)
 33 C,  50% humidity -> fan  82.8%   (slow=0.00, medium=0.00, fast=0.80)


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
  8 C,  30% humidity -> fan  17.2%   (slow=0.80, medium=0.00, fast=0.00)
 21 C,  30% humidity -> fan  50.0%   (slow=0.00, medium=0.86, fast=0.00)
 24 C,  75% humidity -> fan  66.5%   (slow=0.00, medium=0.75, fast=0.75)
 33 C,  50% humidity -> fan  82.8%   (slow=0.00, medium=0.00, fast=0.80)
```

**What the output shows**

Compare the output for 21 C and 24 C at high humidity. The fan speed changes smoothly rather than jumping between fixed settings, which is exactly why fuzzy control produces gentle, human-like behaviour.

**Investigate**

1. Plot fan speed for every temperature from 0 to 45 C at 30% and at 80% humidity using matplotlib. Describe the shape of the curves.
2. Replace the centroid defuzzification with "mean of maximum" (average speed where combined is highest). How does the output change?
3. Rewrite the controller as crisp if-statements (below 18 slow, 18 to 28 medium, above 28 fast) and plot both. Which gives better behaviour for a real fan?

## 9. A two-node Bayesian network (Flu -> Fever)

**File:** `l1_flu_fever.py`

In [10]:
"""
Layer 1 example: a two-node Bayesian network (Flu -> Fever)
File: l1_flu_fever.py
 
Purpose:
  Shows the two directions of probabilistic reasoning in a Bayesian network,
  the smallest possible one, so every number can be checked by hand:
    prediction  (cause -> effect):  how likely is a fever?
    diagnosis   (effect -> cause):  given a fever, how likely is flu?
 
How it works:
  1. Build the graph: a single arrow from Flu to Fever.
  2. Attach probability tables (CPDs): the prior for Flu, and P(Fever | Flu).
  3. Ask pgmpy to infer P(Fever) and P(Flu | Fever = yes).
  4. Repeat both calculations by hand with Bayes' theorem.
 
What to look for in the output:
  - P(Fever = yes) is 0.14, mixing true fevers (flu) and false alarms.
  - Seeing a fever raises belief in flu from 5% to only about 32%, because
    flu is rare: most fevers come from the much larger no-flu group.
  - The hand calculation matches pgmpy exactly.
 
Requires: pip install pgmpy   (tested with pgmpy 1.1)
"""
import logging, warnings
# hide pgmpy deprecation notices
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("pgmpy").setLevel(logging.ERROR)   # hide pgmpy's info messages
 
# pgmpy 1.x names the class DiscreteBayesianNetwork; older versions used BayesianNetwork
try:
    from pgmpy.models import DiscreteBayesianNetwork as BN
except ImportError:
    from pgmpy.models import BayesianNetwork as BN
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
 
# ---- 1. The graph: one arrow, from cause to effect -------------------------
# Flu is the parent of Fever: having flu directly changes the chance of fever.
model = BN([("Flu", "Fever")])
 
# ---- 2. The probability tables (CPDs) --------------------------------------
# Flu has no parents, so its table is just the base rate (the "prior").
# State 0 = no, state 1 = yes. Each column must add up to 1.
cpd_flu = TabularCPD(
    variable="Flu", variable_card=2,          # card = number of states
    values=[[0.95],                           # P(Flu = no)
            [0.05]],                          # P(Flu = yes): 5% of patients
    state_names={"Flu": ["no", "yes"]},
)
 
# Fever's table has one column per value of its parent, Flu.
cpd_fever = TabularCPD(
    variable="Fever", variable_card=2,
    # Row 0: P(Fever = no  | Flu = no),  P(Fever = no  | Flu = yes)
    # Row 1: P(Fever = yes | Flu = no),  P(Fever = yes | Flu = yes)
    values=[[0.90, 0.10],
            [0.10, 0.90]],
    evidence=["Flu"], evidence_card=[2],
    state_names={"Fever": ["no", "yes"], "Flu": ["no", "yes"]},
)
 
model.add_cpds(cpd_flu, cpd_fever)
# checks every column sums to 1 and the graph matches the tables
assert model.check_model()
 
print("Probability tables")
print(cpd_flu)
print(cpd_fever)
 
# ---- 3. Ask questions -------------------------------------------------------
infer = VariableElimination(model)
 
# Prediction: no evidence at all, so what is the overall chance of a fever?
p_fever = infer.query(["Fever"], show_progress=False)
print("\nPrediction: P(Fever) with no evidence")
print(p_fever)
 
# Diagnosis: we observe a fever, so how likely is flu now?
p_flu = infer.query(["Flu"], evidence={"Fever": "yes"}, show_progress=False)
print("\nDiagnosis: P(Flu | Fever = yes)")
print(p_flu)
 
# ---- 4. Check the answers by hand with Bayes' theorem ----------------------
prior, hit, false_alarm = 0.05, 0.90, 0.10
fever = prior * hit + (1 - prior) * false_alarm          # law of total probability
posterior = prior * hit / fever                          # Bayes' theorem
print(f"\nBy hand: P(Fever) = {fever:.3f}, P(Flu | Fever) = {posterior:.3f}")
print(f"Prior belief in flu {prior:.0%} -> after seeing a fever {posterior:.0%}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Probability tables


+----------+------+
| Flu(no)  | 0.95 |
+----------+------+
| Flu(yes) | 0.05 |
+----------+------+
+------------+---------+----------+
| Flu        | Flu(no) | Flu(yes) |
+------------+---------+----------+
| Fever(no)  | 0.9     | 0.1      |
+------------+---------+----------+
| Fever(yes) | 0.1     | 0.9      |
+------------+---------+----------+

Prediction: P(Fever) with no evidence
+------------+--------------+
| Fever      |   phi(Fever) |
+============+==============+
| Fever(no)  |       0.8600 |
+------------+--------------+
| Fever(yes) |       0.1400 |
+------------+--------------+

Diagnosis: P(Flu | Fever = yes)
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(no)  |     0.6786 |
+----------+------------+
| Flu(yes) |     0.3214 |
+----------+------------+

By hand: P(Fever) = 0.140, P(Flu | Fever) = 0.321
Prior belief in flu 5% -> after seeing a fever 32%


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Probability tables
+----------+------+
| Flu(no)  | 0.95 |
+----------+------+
| Flu(yes) | 0.05 |
+----------+------+
+------------+---------+----------+
| Flu        | Flu(no) | Flu(yes) |
+------------+---------+----------+
| Fever(no)  | 0.9     | 0.1      |
+------------+---------+----------+
| Fever(yes) | 0.1     | 0.9      |
+------------+---------+----------+
Prediction: P(Fever) with no evidence
+------------+--------------+
| Fever      |   phi(Fever) |
+============+==============+
| Fever(no)  |       0.8600 |
+------------+--------------+
| Fever(yes) |       0.1400 |
+------------+--------------+
Diagnosis: P(Flu | Fever = yes)
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(no)  |     0.6786 |
+----------+------------+
| Flu(yes) |     0.3214 |
+----------+------------+
By hand: P(Fever) = 0.140, P(Flu | Fever) = 0.321
Prior belief in flu 5% -> after seeing a fever 32%
```